v62 
- convert codebase for RL agent  
- headless scorer
- cross-sectional normalization
- discovery composite action
- temporal vectorizer

v61 
- **Verified all metric calculation with Excel** 
- Updated core.analyzer
- Replaced the `Result` pattern with exceptions and flattened the logic
- Refactored the `AlphaEngine` into a streamlined Orchestrator pattern
-  **Strict Date Logic:** No more "Time Travel" bugs.
-  **Dataclass Contracts:** No more "Magic String" typos or blind dictionaries.
-  **Exception Flow:** The `run` method is now a clean, readable story instead of a maze of `if/else` checks.
-  **Performance Workers:** Math is separated from orchestration.
- Ret_1d, explicitly turns division-by-zero results (`inf`) into `NaN`, and replace [np.inf, -np.inf] with np.nan



v60  
- Converted code from notebook to modular system.
- Fixed divide by zero warning from calculate_gain
- Added subtitle to subplots
- Added Volatility Regime plot


v59  
- Removed "nest" of if-statements in **AlphaEngine.run**
- Use **Result Pattern** to handle errors
- Change verify_analyzer_short and verify_analyzer_long gain calculation from simple return to logarithmic return
- Change calculate_gain from simple return to logarithmic return
- Remove bfill from calculate_gain to prevent backfill with future data
- Verify macro_df calculation


In [72]:
# 1. Enable Autoreload
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

def add_project_root_to_path():
    """Find notebooks_RLVR and add to sys.path."""
    current = Path.cwd()

    # Search upward for notebooks_RLVR folder
    for path in [current] + list(current.parents):
        if path.name == "notebooks_RLVR":
            sys.path.insert(0, str(path))
            print(f"✓ Added to path: {path}")
            return path
        # Also check if notebooks_RLVR exists as child (for running from stocks/)
        candidate = path / "notebooks_RLVR"
        if candidate.exists():
            sys.path.insert(0, str(candidate))
            print(f"✓ Added to path: {candidate}")
            return candidate

    raise RuntimeError("Could not find notebooks_RLVR directory")


# Run once at notebook start
add_project_root_to_path()


# 2. Force reload cached modules (run this to refresh code changes)
modules_to_reload = [
    "core.analyzer",
    "core.auditor",
    "core.contracts",    
    "core.engine",
    "core.features",
    "core.paths",
    "core.performance",
    "core.quant",
    "core.result",
    "core.settings",
    "core.utils",
    "strategy.registry",
]

for mod in modules_to_reload:
    if mod in sys.modules:
        del sys.modules[mod]


# 3. Standard imports
import pandas as pd
import numpy as np

from IPython.display import display


# 4. Fresh imports (these will re-import from disk due to cache clearing above)
from core.quant import QuantUtils
from core.analyzer import create_walk_forward_analyzer
from core.auditor import SystemAuditor as SA
from core.contracts import FilterPack, EngineInput
from core.engine import AlphaEngine
from core.features import generate_features
from core.paths import OUTPUT_DIR
from core.settings import GLOBAL_SETTINGS
from core.utils import SystemUtils as SU
from strategy.registry import METRIC_REGISTRY


# 5. Pandas display settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.precision", 4)


# 6. Load data path from .env
DATA_PATH_OHLCV = SU.load_env_from_root("DATA_PATH_OHLCV")
DATA_PATH_INDICES = SU.load_env_from_root("DATA_PATH_INDICES")
print(f"DATA_PATH_OHLCV: {DATA_PATH_OHLCV}")
print(f"DATA_PATH_INDICES: {DATA_PATH_INDICES}")

# 7. Instantiate engine (customize DataFrames as needed)
master_engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✓ Added to path: c:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR
NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR

OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output

DATA_PATH_OHLCV: c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet
DATA_PATH_INDICES: c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices.parquet


In [2]:
df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
print(f"df_indices:|n{df_indices}")

df_indices:|n                   Adj Open  Adj High  Adj Low  Adj Close  Volume
Ticker Date                                                      
^AXJO  1992-11-22   1455.00   1455.00  1455.00    1455.00       0
       1992-11-23   1458.40   1458.40  1458.40    1458.40       0
       1992-11-24   1467.90   1467.90  1467.90    1467.90       0
       1992-11-25   1459.00   1459.00  1459.00    1459.00       0
       1992-11-26   1458.90   1458.90  1458.90    1458.90       0
...                     ...       ...      ...        ...     ...
^VIX3M 2026-03-09     28.04     29.09    24.86      25.34       0
       2026-03-10     24.94     25.84    23.54      25.51       0
       2026-03-11     25.29     25.98    24.93      24.97       0
       2026-03-12     26.27     26.99    25.80      26.95       0
       2026-03-13     25.92     27.34    25.51      27.28       0

[144750 rows x 5 columns]


In [3]:
_indices = df_indices.index.get_level_values(0).unique().tolist()
display(_indices)
df_indices.info()

['^AXJO',
 '^BSESN',
 '^DJI',
 '^FCHI',
 '^FTSE',
 '^GDAXI',
 '^GSPC',
 '^HSI',
 '^IXIC',
 '^N225',
 '^NYA',
 '^STOXX50E',
 '^VIX',
 '^VIX3M']

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 144750 entries, ('^AXJO', Timestamp('1992-11-22 00:00:00')) to ('^VIX3M', Timestamp('2026-03-13 00:00:00'))
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   Adj Open   144750 non-null  float64
 1   Adj High   144750 non-null  float64
 2   Adj Low    144750 non-null  float64
 3   Adj Close  144750 non-null  float64
 4   Volume     144750 non-null  int64  
dtypes: float64(4), int64(1)
memory usage: 6.6+ MB


In [4]:
print(f"Takes about 1.5 minutes")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")

Takes about 1.5 minutes


In [5]:
# print(f"df_ohlcv.head():\n {df_ohlcv.head()}\n")
df_ohlcv.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 9541610 entries, ('A', Timestamp('1999-11-18 00:00:00')) to ('ZWS', Timestamp('2026-03-13 00:00:00'))
Data columns (total 5 columns):
 #   Column     Dtype  
---  ------     -----  
 0   Adj Open   float64
 1   Adj High   float64
 2   Adj Low    float64
 3   Adj Close  float64
 4   Volume     int64  
dtypes: float64(4), int64(1)
memory usage: 401.1+ MB


In [6]:
print(f"Takes about 3 minutes to generate_features")

features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv,
    df_indices=df_indices,
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
)

Takes about 3 minutes to generate_features
⚡ Generating Decoupled Features (Benchmark: SPY)...


In [7]:
print(f"features_df.info():\n{features_df.info()}\n")
print(f"features_df.index.names:\n{features_df.index.names}\n")
print(f"macro_df.info():\n{macro_df.info()}\n")
print(f"macro_df.index.names:\n{macro_df.index.names}\n")

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 9541610 entries, ('A', Timestamp('1999-11-18 00:00:00')) to ('ZWS', Timestamp('2026-03-13 00:00:00'))
Data columns (total 13 columns):
 #   Column               Dtype  
---  ------               -----  
 0   ATR                  float64
 1   ATRP                 float64
 2   TRP                  float64
 3   RSI                  float64
 4   Mom_21               float64
 5   Consistency          float64
 6   IR_63                float64
 7   Beta_63              float64
 8   DD_21                float64
 9   Ret_1d               float64
 10  RollingStalePct      float64
 11  RollMedDollarVol     float64
 12  RollingSameVolCount  float64
dtypes: float64(13)
memory usage: 983.5+ MB
features_df.info():
None

features_df.index.names:
['Ticker', 'Date']

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 16157 entries, 1962-01-02 to 2026-03-13
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------      

In [8]:
# Pre-flight Automated Audit Suite
checks = [
    SA.verify_math_integrity(),
    SA.verify_ranking_integrity(),
    SA.verify_vol_alignment_integrity(),
    SA.verify_feature_engineering_integrity(),
]

for check in checks:
    icon = "✅" if check.ok else "🔥"
    print(f"{icon} {check.msg}")
    if not check.ok:
        print("🛑 Critical Failure. System Halted.")
        break

print("=" * 85)
# Separate verify_marco_engine output from intertwine with other outputs

checks = [
    SA.verify_macro_engine(
        df_ohlcv=df_ohlcv,
        df_indices=df_indices,
        original_macro_df=macro_df,
        settings=GLOBAL_SETTINGS,
    ),
]

for check in checks:
    icon = "✅" if check.ok else "🔥"
    print(f"{icon} {check.msg}")
    if not check.ok:
        print("🛑 Critical Failure. System Halted.")
        break


--- 🛡️ Starting Feature Engineering Audit ---
⚡ Generating Decoupled Features (Benchmark: SPY)...
Audit Values:
[ nan 25.  17.5]
✅ FEATURE INTEGRITY PASSED: Wilder's ATR logic is strictly enforced.
✅ Mathematical boundaries strictly enforced.
✅ Ranking integrity strictly enforced.
✅ Reward and Risk are strictly synchronized.
✅ Wilder's ATR logic is strictly enforced.
--- Macro Verification (Benchmark: SPY) ---

Comparing verification vs original (Clip Threshold: 4.0):
✅ Mkt_Ret              | PASS (Max Diff: 0.00e+00)
✅ Macro_Trend          | PASS (Max Diff: 0.00e+00)
✅ Macro_Trend_Vel      | PASS (Max Diff: 0.00e+00)
✅ Macro_Trend_Vel_Z    | PASS (Max Diff: 0.00e+00)
✅ Macro_Trend_Mom      | PASS (Max Diff: 0.00e+00)
✅ Macro_Vix_Z          | PASS (Max Diff: 0.00e+00)
✅ Macro_Vix_Ratio      | PASS (Max Diff: 0.00e+00)
✅ Macro Integrity Verified


In [9]:
print(
    "🚀 Generating Wide Matrices for Instant Backtesting... (takes about 1 minute to run)"
)

# 1. Price Matrix
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)

# 2. Volatility Matrices (Unstack and Align)
# Using features_df (the first item from the tuple)
print("   - Unstacking ATRP...")
df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)

print("   - Unstacking TRP...")
df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)

# 3. Handle Data Gaps (Sanitize the Wide Matrices)
if GLOBAL_SETTINGS["handle_zeros_as_nan"]:
    df_close_wide = df_close_wide.replace(0, np.nan)

# Forward fill up to the limit, then fill remaining with the "Disaster Detection" value
df_close_wide = df_close_wide.ffill(limit=GLOBAL_SETTINGS["max_data_gap_ffill"])
df_close_wide = df_close_wide.fillna(GLOBAL_SETTINGS["nan_price_replacement"])

print(
    f"✅ Pre-computation Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)
print("   Ready: df_close_wide, df_atrp_wide, df_trp_wide, and macro_df.")

🚀 Generating Wide Matrices for Instant Backtesting... (takes about 1 minute to run)
   - Unstacking ATRP...
   - Unstacking TRP...
✅ Pre-computation Complete. Tickers: 1588, Days: 16157
   Ready: df_close_wide, df_atrp_wide, df_trp_wide, and macro_df.


In [45]:
# 6. Instantiate engine (customize DataFrames as needed)
master_engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)

In [73]:
decision_dt = pd.Timestamp("2025-12-10")  # Pick a known date
lookback = 10

# A. Run via current UI logic (Manual)
# (Imagine you selected 'Sharpe (ATRP)' in the dropdown)
ui_result = master_engine.run(
    EngineInput(
        mode="Rank",
        start_date=decision_dt,
        lookback_period=lookback,
        holding_period=5,
        metric="Sharpe (ATRP)",
        benchmark_ticker="SPY",
    )
)
ui_scores = ui_result.results_df["Strategy Value"]

# B. Run via new Headless Scorer
alpha_matrix = master_engine.compute_alpha_matrix(decision_dt, lookback)
headless_scores = alpha_matrix["Sharpe (ATRP)"].reindex(ui_scores.index)

# VERIFICATION
pd.testing.assert_series_equal(ui_scores, headless_scores, check_names=False)
print("✅ Step 1 Success: Headless Scorer matches UI logic perfectly.")

DEBUG: 940 stocks passed filters on 2025-12-10
✅ Step 1 Success: Headless Scorer matches UI logic perfectly.


In [74]:
# --- TEST 2: NORMALIZATION INTEGRITY ---
raw_matrix = master_engine.compute_alpha_matrix(decision_dt, lookback)
norm_matrix = master_engine.normalize_alpha_matrix(raw_matrix)

# Verification A: Mean should be near 0
mean_check = norm_matrix.mean().abs().max()
# Verification B: Std should be near 1
std_check = norm_matrix.std().max()

print(f"Max Mean Offset: {mean_check:.6f} (Target: < 0.01)")
print(f"Max Std: {std_check:.6f} (Target: ~ 1.0)")

if mean_check < 0.01 and 0.8 < std_check < 1.2:
    print("✅ Step 2 Success: The Alpha Matrix is now 'Agent-Ready'.")
else:
    print("❌ Step 2 Failed: Normalization drift detected.")

Max Mean Offset: 0.009367 (Target: < 0.01)
Max Std: 1.000000 (Target: ~ 1.0)
✅ Step 2 Success: The Alpha Matrix is now 'Agent-Ready'.


In [75]:
# --- TEST 2.1: REGIME AWARENESS VERIFICATION ---
# decision_dt = pd.Timestamp("2025-12-10")
context = master_engine.compute_context_vector(decision_dt)

print("--- Agent Macro Context ---")
print(context)

# Check against your Plotly Image (Dec 10 values):
# Trend (Green line): Should be ~10% (0.10)
# Trend Vel (Orange line): Should be slightly below 0.0
# VIX Ratio: Chart says 0.86
# VIX Z (Purple line): Should be around -1.0

print(f"\nVerification Check:")
print(f"VIX Ratio in Context: {context['Context_Vix_Ratio'] + 1:.2f} (Target: 0.86)")

--- Agent Macro Context ---
Context_Trend        1.1401
Context_Vel_Z       -0.3619
Context_Vix_Z       -0.8380
Context_Vix_Ratio   -0.1850
dtype: float64

Verification Check:
VIX Ratio in Context: 0.81 (Target: 0.86)


In [ ]:
# --- TEST 3 (REVISITED): VERBOSE DISCOVERY VERIFICATION ---
decision_dt = pd.Timestamp("2025-12-10")
lookback = 21

# 1. Setup 'One-Hot' Action for Sharpe (ATRP)
registry_keys = list(METRIC_REGISTRY.keys())
action_weights = np.zeros(len(registry_keys))
action_weights[registry_keys.index("Sharpe (ATRP)")] = 1.0

# 2. Run Discovery
discovery = master_engine.run_discovery_action(
    decision_dt, lookback, holding_period=5, weights=action_weights
)

# 3. Get UI Result for verification
ui_result = master_engine.run(
    EngineInput(
        mode="Ranking",
        start_date=decision_dt,
        lookback_period=lookback,
        holding_period=5,
        metric="Sharpe (ATRP)",
        benchmark_ticker="SPY",
    )
)

print(f"=== DISCOVERY ENGINE VERIFICATION (Date: {decision_dt.date()}) ===")
print(f"Target Strategy Weights: {discovery.action_weights}")
print(f"Selected Tickers: {discovery.selected_tickers}")
print("-" * 50)
print(f"Internal Discovery Score (Top 1): {discovery.metric_values.iloc[0]:.4f}")
print(
    f"Original UI Strategy Value (Top 1): {ui_result.results_df['Strategy Value'].iloc[0]:.4f}"
)
print("-" * 50)
print(f"VERITABLE REWARD (Sharpe): {discovery.veritable_reward:.4f}")
print(f"UI REWARD (Sharpe):        {ui_result.perf_metrics['holding_p_sharpe']:.4f}")

if discovery.selected_tickers == ui_result.tickers:
    print("\n✅ SELECTION MATCH: The agent and UI chose the same tickers.")

In [77]:
discovery

DiscoveryResult(action_weights={'Price Gain': 0.0, 'Sharpe': 0.0, 'Sharpe (ATRP)': 1.0, 'Sharpe (TRP)': 0.0, 'Momentum (21d)': 0.0, 'Info Ratio (Stdev_Alpha, 63d)': 0.0, 'Consistency (WinRate 5d)': 0.0, 'Oversold (-RSI)': 0.0, 'Dip Buyer (Drawdown -dd_21)': 0.0, 'Low Volatility (-ATRP)': 0.0, 'VIX Filtered Momentum': 0.0}, selected_tickers=['SHV', 'BIL', 'EXAS', 'MINT', 'SGOV', 'BOXX', 'USFR', 'PULS', 'RY', 'DG'], veritable_reward=0.003709274626975093, metric_values=Ticker
SHV     4.0000
BIL     4.0000
EXAS    4.0000
MINT    4.0000
SGOV    4.0000
BOXX    3.5340
USFR    3.3304
PULS    3.0036
RY      2.8311
DG      2.6402
dtype: float64)

In [78]:
# --- STEP 4.1: THE VERITABLE STANDARD PROOF ---

# 1. Setup Parameters
decision_dt = pd.Timestamp("2025-12-10")
holding_pd = 5
lookback = 21
ticker = "SHV"  # Let's test with the ticker you verified

# 2. METHOD A: The Verified UI Engine (Compounded Daily Returns)
# This is the code you already verified with Excel
ui_result = master_engine.run(
    EngineInput(
        mode="Manual List",
        start_date=decision_dt,
        lookback_period=lookback,
        holding_period=holding_pd,
        metric="Price Gain",
        manual_tickers=[ticker],
        benchmark_ticker="SPY",
    )
)
buy_date = ui_result.buy_date
end_date = ui_result.holding_end_date
ui_gain = ui_result.perf_metrics["holding_p_gain"]

# 3. METHOD B: The New Vectorized Shifter (Price Ratio)
# We calculate the log-gain directly from the price matrix
# log(P_end / P_start)
price_start = master_engine.df_close.loc[buy_date, ticker]
price_end = master_engine.df_close.loc[end_date, ticker]
vectorized_gain = np.log(price_end / price_start)

# 4. FINAL COMPARISON
print(f"=== VERITABLE STANDARD PROOF ({ticker}) ===")
print(f"Buy Date: {buy_date.date()} | End Date: {end_date.date()}")
print(f"Price at Buy: {price_start:.4f}")
print(f"Price at End: {price_end:.4f}")
print("-" * 40)
print(f"Method A (UI Engine Gain):   {ui_gain:.8f}")
print(f"Method B (Shifted Matrix):   {vectorized_gain:.8f}")
print("-" * 40)

diff = abs(ui_gain - vectorized_gain)
if diff < 1e-10:
    print("✅ VERIFICATION SUCCESS: Both methods are mathematically identical.")
    print("The Vectorized 'Time Machine' is now safe to use for RL Training.")
else:
    print(f"❌ VERIFICATION FAILED: Difference of {diff:.10f} detected.")

#

=== VERITABLE STANDARD PROOF (SHV) ===
Buy Date: 2025-12-11 | End Date: 2025-12-18
Price at Buy: 109.2710
Price at End: 109.3600
----------------------------------------
Method A (UI Engine Gain):   0.00081416
Method B (Shifted Matrix):   0.00081416
----------------------------------------
✅ VERIFICATION SUCCESS: Both methods are mathematically identical.
The Vectorized 'Time Machine' is now safe to use for RL Training.


In [79]:
# --- STEP 4.2: VECTORIZED DISCOVERY VERIFICATION ---

# A. Initialize the Time Machine (Do this once)
master_engine.precompute_reward_matrix(holding_period=5)

decision_dt = pd.Timestamp("2025-12-10")
lookback = 21

# B. Setup Action (Sharpe ATRP)
registry_keys = list(METRIC_REGISTRY.keys())
weights = np.zeros(len(registry_keys))
weights[registry_keys.index("Sharpe (ATRP)")] = 1.0

# C. Run Discovery (Using the NEW Vectorized functions)
raw_matrix, discovery = master_engine.run_discovery_action(
    decision_dt, lookback, holding_period=5, weights=weights
)

# D. Get UI Result for the "Gold Standard"
ui_result = master_engine.run(
    EngineInput(
        mode="Ranking",
        start_date=decision_dt,
        lookback_period=lookback,
        holding_period=5,
        metric="Sharpe (ATRP)",
        benchmark_ticker="SPY",
    )
)

print(f"=== VECTORIZED ENGINE VERIFICATION (Date: {decision_dt.date()}) ===")
print(
    f"Selected Tickers: {discovery.selected_tickers[:3]}... (Match UI: {discovery.selected_tickers == ui_result.tickers})"
)
print("-" * 60)

# This confirms the 0.8381 'Signal' is preserved
print(f"SIGNAL CHECK (Lookback):")
print(f"  Discovery Score (Top 1):    {discovery.metric_values.iloc[0]:.4f}")
print(
    f"  Original UI Strategy Val:   {ui_result.results_df['Strategy Value'].iloc[0]:.4f}"
)

print("-" * 60)

# This confirms the 'Reward' matches the price action
# Note: UI Reward is Sharpe (28.29), Discovery Reward is now Total Return for RL efficiency
ui_return = ui_result.perf_metrics["holding_p_gain"]

print(f"REWARD CHECK (Holding Period Return):")
print(f"  Vectorized Reward:          {discovery.veritable_reward:.8f}")
print(f"  UI Verified Gain:           {ui_return:.8f}")

if abs(discovery.veritable_reward - ui_return) < 1e-7:
    print("\n✅ VERITABLE MATCH: The Time Machine matches the UI reality.")
    print("The Engine is now ready for High-Frequency Training.")



raw_matrix:
        Price Gain  Sharpe  Sharpe (ATRP)  Sharpe (TRP)  Momentum (21d)  Info Ratio (Stdev_Alpha, 63d)  Consistency (WinRate 5d)  Oversold (-RSI)  Dip Buyer (Drawdown -dd_21)  Low Volatility (-ATRP)  VIX Filtered Momentum
Ticker                                                                                                                                                                                                                       
A          -0.0199 -0.6780        -0.0299       -0.0280         -0.0197                         0.0818                       0.2         -44.3911                       0.0873                 -0.0263                -0.0197
AA          0.1405  3.3138         0.1597        0.1590          0.1508                         0.1564                       0.4         -66.9777                      -0.0000                 -0.0456                 0.1508
AAL         0.1288  4.3352         0.1555        0.1733          0.1375                         0.

In [ ]:
raw_matrix.loc["SHV"]

Price Gain                        0.0033
Sharpe                           23.6276
Sharpe (ATRP)                     0.8411
Sharpe (TRP)                      0.8176
Momentum (21d)                    0.0033
Info Ratio (Stdev_Alpha, 63d)    -0.0820
Consistency (WinRate 5d)          0.8000
Oversold (-RSI)                 -98.9555
Dip Buyer (Drawdown -dd_21)      -0.0000
Low Volatility (-ATRP)           -0.0002
VIX Filtered Momentum             0.0033
Name: SHV, dtype: float64

In [81]:
discovery

DiscoveryResult(action_weights={'Price Gain': 0.0, 'Sharpe': 0.0, 'Sharpe (ATRP)': 1.0, 'Sharpe (TRP)': 0.0, 'Momentum (21d)': 0.0, 'Info Ratio (Stdev_Alpha, 63d)': 0.0, 'Consistency (WinRate 5d)': 0.0, 'Oversold (-RSI)': 0.0, 'Dip Buyer (Drawdown -dd_21)': 0.0, 'Low Volatility (-ATRP)': 0.0, 'VIX Filtered Momentum': 0.0}, selected_tickers=['BIL', 'SHV', 'SGOV', 'EXAS', 'MINT', 'BOXX', 'USFR', 'PULS', 'RY', 'DG'], veritable_reward=0.003709274626975093, metric_values=Ticker
BIL     4.0000
SHV     4.0000
SGOV    4.0000
EXAS    4.0000
MINT    4.0000
BOXX    3.5376
USFR    3.3331
PULS    3.0139
RY      2.8091
DG      2.6374
dtype: float64)

In [24]:
# ui_result.perf_metrics
# registry_keys
# sharpe_atrp_idx
action_weights

array([0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.])

In [15]:
macro_df.loc[decision_dt]

Mkt_Ret              0.0066
Macro_Trend          0.1140
Macro_Trend_Vel     -0.0047
Macro_Trend_Vel_Z   -0.3619
Macro_Trend_Mom     -0.0047
Macro_Vix_Z         -0.8380
Macro_Vix_Ratio      0.8150
Name: 2025-12-10 00:00:00, dtype: float64

In [82]:
# universe_subset=None means "Scan the whole market"
analyzer1, stage1_pack = create_walk_forward_analyzer(
    master_engine, universe_subset=None
)

print("🚀 Ready for Stage 1: Run Simulation for first filter.")
analyzer1.show()

🚀 Ready for Stage 1: Run Simulation for first filter.
DEBUG: 940 stocks passed filters on 2025-12-10


In [17]:
# tick_price_252 = analyzer1.last_run.tickers
# tick_price_189 = analyzer1.last_run.tickers
# tick_price_126 = analyzer1.last_run.tickers
# tick_price_63 = analyzer1.last_run.tickers
# tick_price_42 = analyzer1.last_run.tickers
# tick_price_21 = analyzer1.last_run.tickers
# tick_price_10 = analyzer1.last_run.tickers

In [18]:
# tick_sharpe_atrp_252 = analyzer1.last_run.tickers
# tick_sharpe_atrp_189 = analyzer1.last_run.tickers
# tick_sharpe_atrp_126 = analyzer1.last_run.tickers
# tick_sharpe_atrp_63 = analyzer1.last_run.tickers
# tick_sharpe_atrp_42 = analyzer1.last_run.tickers
# tick_sharpe_atrp_21 = analyzer1.last_run.tickers
# tick_sharpe_atrp_10 = analyzer1.last_run.tickers

In [ ]:
union, intersection = set_operations(
    tick_sharpe_atrp_252,
    tick_sharpe_atrp_189,
    tick_sharpe_atrp_126,
    tick_sharpe_atrp_63,
    tick_sharpe_atrp_42,
    tick_sharpe_atrp_21,
    tick_sharpe_atrp_10,
)

print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

int_sharpe_atrp = intersection
# int_252_189_126 = intersection

In [ ]:
print(list_to_string(int_sharpe_atrp))

In [ ]:
def list_to_string(items, separator=", ", last_separator=None):
    """
    Convert list to string with customizable separators

    Parameters:
    -----------
    items : list of strings
    separator : str, default ', '
        Separator between items
    last_separator : str, optional
        Special separator for last item (e.g., ' and ', ' or ')

    Returns:
    --------
    str : Formatted string

    Examples:
    ---------
    list_to_string(['a', 'b', 'c'])              # "a, b, c"
    list_to_string(['a', 'b', 'c'], ' | ')        # "a | b | c"
    list_to_string(['a', 'b', 'c'], ', ', ' and ')  # "a, b and c"
    """
    if not items:
        return ""

    if len(items) == 1:
        return str(items[0])

    if last_separator and len(items) > 1:
        return separator.join(items[:-1]) + last_separator + items[-1]

    return separator.join(str(item) for item in items)


# Usage examples
print(list_to_string(["a", "b", "c"]))  # a, b, c
print(list_to_string(["a", "b", "c"], " | "))  # a | b | c
print(list_to_string(["a", "b", "c"], ", ", " and "))  # a, b and c
print(list_to_string(["a", "b", "c"], ", ", ", and "))  # a, b, and c
print(list_to_string(["apple", "banana"], ", ", " and "))  # apple and banana

In [ ]:
union, intersection = set_operations(
    tick_price_252,
    tick_price_189,
    tick_price_126,
    tick_price_63,
    tick_price_42,
    tick_price_21,
    tick_price_10,
)

print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

int_price = intersection
# int_252_189_126 = intersection

In [ ]:
union, intersection = set_operations(
    int_sharpe_atrp,
    int_price,
)
print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

int_shp_atrp_price = intersection

In [ ]:
def set_operations(*lists):
    """
    Find sorted union, intersection, and elements unique to first list

    Parameters:
    -----------
    *lists : variable number of lists of strings

    Returns:
    --------
    tuple : (sorted_union, sorted_intersection, unique_to_first_list)
        - union: all unique elements from all lists
        - intersection: common elements in ALL lists
        - unique_to_first: elements only in first list, not in any other list

    Examples:
    ---------
    union, intersection, unique_first = set_operations(list1, list2)
    union, intersection, unique_first = set_operations(list1, list2, list3, list4)
    """

    if not lists:
        return [], [], []

    # Convert each list to a set
    sets = [set(lst) for lst in lists]
    first_set = sets[0]
    other_sets = sets[1:] if len(sets) > 1 else []

    # Union: all unique elements from all lists
    union_set = set().union(*sets)
    union = sorted(union_set)

    # Intersection: common elements in ALL lists
    intersection_set = first_set.intersection(*other_sets) if other_sets else first_set
    intersection = sorted(intersection_set)

    # Unique to first list: in first_set but NOT in any other set
    # Using difference: first_set - (union of all other sets)
    if other_sets:
        all_others = set().union(*other_sets)
        unique_to_first_set = (
            first_set - all_others
        )  # or first_set.difference(all_others)
    else:
        unique_to_first_set = (
            first_set  # If only one list, all elements are "unique" to it
        )

    unique_to_first = sorted(unique_to_first_set)

    return union, intersection, unique_to_first


#

In [ ]:
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# 1. LAUNCH STAGE 2 (Cascade)
# universe_subset=analyzer1.last_run.tickers means "Scan the whole market"
analyzer2, stage1_pack = create_walk_forward_analyzer(
    master_engine,
    universe_subset=analyzer1.last_run.tickers,
    # universe_subset=None,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

In [ ]:
###############################
###############################

In [85]:
my_analyzer = analyzer1

my_res = SU.visualize_analyzer_structure(my_analyzer)

🔍 HIGH-TRANSPARENCY AUDIT MAP
[  0] 📦 audit_pack (EngineOutput)
[  1]   📈 portfolio_series (shape=(17,))
[  2]   📈 benchmark_series (shape=(17,))
[  3]   🧮 normalized_plot_data (shape=(17, 5))
[  4]   📂 tickers (len=5)
[  5]     📄 index_0 (str)
[  6]     📄 index_1 (str)
[  7]     📄 index_2 (str)
[  8]     📄 index_3 (str)
[  9]     📄 index_4 (str)
[ 10]   📈 initial_weights (shape=(5,))
[ 11]   📂 perf_metrics (len=24)
[ 12]     🔢 full_p_gain (float)
[ 13]     🔢 full_p_sharpe (float)
[ 14]     🔢 full_p_sharpe_atrp (float)
[ 15]     🔢 full_p_sharpe_trp (float)
[ 16]     🔢 lookback_p_gain (float)
[ 17]     🔢 lookback_p_sharpe (float)
[ 18]     🔢 lookback_p_sharpe_atrp (float)
[ 19]     🔢 lookback_p_sharpe_trp (float)
[ 20]     🔢 holding_p_gain (float)
[ 21]     🔢 holding_p_sharpe (float)
[ 22]     🔢 holding_p_sharpe_atrp (float)
[ 23]     🔢 holding_p_sharpe_trp (float)
[ 24]     🔢 full_b_gain (float)
[ 25]     🔢 full_b_sharpe (float)
[ 26]     🔢 full_b_sharpe_atrp (float)
[ 27]     🔢 full_b

In [ ]:
def set_operations(list1, list2):
    """
    Find sorted union and intersection of two lists of strings

    Parameters:
    -----------
    list1, list2 : list of strings

    Returns:
    --------
    tuple : (sorted_union, sorted_intersection)
    """

    # Convert to sets for operations
    set1 = set(list1)
    set2 = set(list2)

    # Union: all unique elements from both lists
    union = sorted(set1 | set2)  # or set1.union(set2)

    # Intersection: common elements in both lists
    intersection = sorted(set1 & set2)  # or set1.intersection(set2)

    return union, intersection


# Example usage
list_a = ["apple", "banana", "cherry", "date", "elderberry"]
list_b = ["banana", "date", "fig", "grape", "apple"]

union, intersection = set_operations(list_a, list_b)

print(f"List 1: {list_a}")
print(f"List 2: {list_b}")
print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

In [ ]:
union, intersection = set_operations(list_a, list_b)

In [ ]:
tick_sharpe_trp_252 = SU.peek(4, my_res)

In [ ]:
analyzer1.last_run.tickers
analyzer1.last_run.start_date
analyzer1.last_run.holding_end_date

In [ ]:
# 3. Post-flight Reconciliation
audit = SA.verify_analyzer_short(my_analyzer)
if not audit.ok:
    print(f"🚨 ALERT: {audit.msg}")
    # You could trigger a notification or log the failure here

In [ ]:
audit = SA.verify_analyzer_long(my_analyzer, n_tickers=5)

In [ ]:
# Takes 4 seconds to run, checks selected tickers from analyzer1
SA.audit_feature_engineering_integrity(my_analyzer, mode="last_run")

### Audit Every Tickers in features_df, Takes 3 minutes 

In [ ]:
# # Takes 3 minutes to run, checks every tickers ~1580 tickers
# SA.audit_feature_engineering_integrity(my_analyzer, mode="system")

### Export Ticker's OHLCV and Features

In [84]:
f_name_excel = OUTPUT_DIR / "Audit_Verification_Report.xlsx"

SU.export_audit_to_excel(audit_pack=my_analyzer.last_run, filename=f_name_excel)

📂 [EXCEL AUDIT] Building full transparency report: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\Audit_Verification_Report.xlsx
✨ Audit Report Complete: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\Audit_Verification_Report.xlsx


WindowsPath('C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR/output/Audit_Verification_Report.xlsx')

In [86]:
f_name_csv = OUTPUT_DIR / "all_tickers_data_stacked.csv"

# Single call replaces your 3 cells
file_path = SU.export_last_run_tickers_data_to_csv(
    analyzer=my_analyzer,
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    filename=f_name_csv,
)

Creating combined dictionary for 6 ticker(s)
Date range: 2025-11-25 00:00:00 to 2025-12-18 00:00:00
Data retrieved for 6 ticker(s) from 2025-11-25 00:00:00 to 2025-12-18 00:00:00
Total rows: 102
Date range in data: 2025-11-25 00:00:00 to 2025-12-18 00:00:00
  SHV: 17 rows
  CFLT: 17 rows
  SGOV: 17 rows
  WBD: 17 rows
  BIL: 17 rows
  SPY: 17 rows
Features data retrieved for 6 ticker(s) from 2025-11-25 00:00:00 to 2025-12-18 00:00:00
Total rows: 102
Date range in data: 2025-11-25 00:00:00 to 2025-12-18 00:00:00
Available features: ATR, ATRP, TRP, RSI, Mom_21, Consistency, IR_63, Beta_63, DD_21, Ret_1d, RollingStalePct, RollMedDollarVol, RollingSameVolCount
  SHV: 17 rows
  CFLT: 17 rows
  SGOV: 17 rows
  WBD: 17 rows
  BIL: 17 rows
  SPY: 17 rows

Processing SHV...
  ✓ Successfully combined data
  OHLCV shape: (17, 5)
  Features shape: (17, 13)
  Combined shape: (17, 18)
  Date range: 2025-11-25 00:00:00 to 2025-12-18 00:00:00

Processing CFLT...
  ✓ Successfully combined data
  OHLCV 

In [ ]:
SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=my_analyzer.last_run.tickers,
    date_start=my_analyzer.last_run.start_date,
    date_end=analyzer1.last_run.holding_end_date,
    verbose=False,
)

In [ ]:
_tic = "NVDA"
_start_date = "2025-03-12"
_end_date = "2026-03-12"
print(features_df.loc[_tic][_start_date:_end_date])
# features_df.loc[_tic][_start_date:_end_date][["Ret_1d", "Consistency"]]

In [ ]:
result = SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=[_tic],
    date_start=_start_date,
    date_end=_end_date,
    verbose=False,
)
print(result[_tic])